In [1]:
import pyspark
from pyspark.sql import SparkSession

## DEFINE SENSITIVE VARIABLES
CATALOG_URI = "http://catalog:19120/api/v1"  # Nessie Server URI
WAREHOUSE = "s3://refine-bucket/warehouse"               # Minio Address to Write to
WAREHOUSE_A = "s3a://refine-bucket/warehouse"               # Minio Address to Write to
STORAGE_URI = "http://minio:9000"      # Minio URL

## SOURCE PARQUET RAW FOLDER
file_star_client = "s3a://raw-bucket/client/*.parquet"
file_star_sales = "s3a://raw-bucket/sales/*.parquet"
file_star_product = "s3a://raw-bucket/product/*.parquet"

In [2]:
"""### !!!!! PACKAGES WILL BE DOWNLOAD ONLINE
# Configure Spark with necessary packages and Iceberg/Nessie settings
conf = (
    pyspark.SparkConf()
        .setAppName('Nessie-Iceberg-Client')
        .set("spark.master", "spark://spark-master:7077") \
        # Include necessary packages
        .set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.99.0,software.amazon.awssdk:bundle:2.24.8,software.amazon.awssdk:url-connection-client:2.24.8')
        # Enable Iceberg and Nessie extensions
        #.set('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
        .set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.nessie-integrations.nessie-spark-extensions-3.4_2.12.NessieSparkSessionExtensions")
        # Configure Nessie catalog
        .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
        .set('spark.sql.catalog.nessie.uri', CATALOG_URI)
        .set('spark.sql.catalog.nessie.ref', 'main')
        .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
        .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
        .set("spark.sql.catalog.nessie.lock-manager-impl", "org.apache.iceberg.aws.s3.S3LockManager")
        .set("spark.sql.catalog.nessie.s3.lock.table", "iceberg_lock_table")
        # Set Minio as the S3 endpoint for Iceberg storage
        .set('spark.sql.catalog.nessie.s3.endpoint', STORAGE_URI)
        .set('spark.sql.catalog.nessie.warehouse', WAREHOUSE_A)
        #.set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
        .set("spark.sql.catalog.nessie.s3.path-style-access", "true")
        # --- Utiliser Hadoop pour les entrées/sorties S3 ---
        .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
        # Set parquet read option for minio (only for raw datas)
        .set("spark.hadoop.fs.s3a.endpoint", STORAGE_URI)
        .set("spark.hadoop.fs.s3a.access.key", "admin")
        .set("spark.hadoop.fs.s3a.secret.key", "password123")
        .set("spark.hadoop.fs.s3a.path.style.access", "true")
        .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        # Optimization
        .set("spark.sql.iceberg.read.split.target-size", "131072")
        .set("spark.executor.memory", "2g")
        .set("spark.executor.cores", "1")
        .set("spark.sql.adaptive.enabled", "true")
        .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .set("spark.sql.iceberg.vectorization.enabled", "false")
        .set("spark.sql.iceberg.distribution-mode", "hash")
        .set("spark.sql.execution.arrow.pyspark.enabled", "true")
)
"""


### !!!! PACKAGES ARE MOUNT IN FOLDER "/home/jovyan/jars"
import os

# Liste des JARs locaux
jars_path = "/home/jovyan/jars"
local_jars = ",".join([os.path.join(jars_path, f) for f in os.listdir(jars_path) if f.endswith('.jar')])
print(local_jars)

# Configure Spark with necessary packages and Iceberg/Nessie settings
conf = (
    pyspark.SparkConf()
        .setAppName('Agence-Nessie-Iceberg-Client')
        .set("spark.master", "spark://spark-master:7077") \
        # load local jar files
        .set("spark.jars", local_jars)
        # Forcer Spark à ne pas chercher sur Internet (Ivy offline)
        .set("spark.io.network.timeout", "60s")
        .set("spark.submit.deployMode", "client")
        .set("spark.jars.ivy.settings", "/dev/null")
        .set("spark.pyspark.python", "python3")
        # Enable Iceberg and Nessie extensions
        #.set('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
        .set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.nessie-integrations.nessie-spark-extensions-3.4_2.12.NessieSparkSessionExtensions")
        # Configure Nessie catalog
        .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
        .set('spark.sql.catalog.nessie.uri', CATALOG_URI)
        .set('spark.sql.catalog.nessie.ref', 'main')
        .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
        .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
        .set("spark.sql.catalog.nessie.lock-manager-impl", "org.apache.iceberg.aws.s3.S3LockManager")
        .set("spark.sql.catalog.nessie.s3.lock.table", "iceberg_lock_table")
        # Set Minio as the S3 endpoint for Iceberg storage
        .set('spark.sql.catalog.nessie.s3.endpoint', STORAGE_URI)
        .set('spark.sql.catalog.nessie.warehouse', WAREHOUSE_A)
        #.set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
        .set("spark.sql.catalog.nessie.s3.path-style-access", "true")
        # --- Utiliser Hadoop pour les entrées/sorties S3 ---
        .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
        # Set parquet read option for minio (only for raw datas)
        .set("spark.hadoop.fs.s3a.endpoint", STORAGE_URI)
        .set("spark.hadoop.fs.s3a.access.key", "admin")
        .set("spark.hadoop.fs.s3a.secret.key", "password123")
        .set("spark.hadoop.fs.s3a.path.style.access", "true")
        .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        # Optimization
        .set("spark.sql.iceberg.read.split.target-size", "131072")
        .set("spark.executor.memory", "1g")
        .set("spark.executor.cores", "1")
        .set("spark.sql.adaptive.enabled", "true")
        .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .set("spark.sql.iceberg.vectorization.enabled", "false")
        .set("spark.sql.iceberg.distribution-mode", "hash")
        .set("spark.sql.execution.arrow.pyspark.enabled", "true")
)


/home/jovyan/jars/org.reactivestreams_reactive-streams-1.0.4.jar,/home/jovyan/jars/software.amazon.awssdk_metrics-spi-2.24.8.jar,/home/jovyan/jars/software.amazon.awssdk_bundle-2.24.8.jar,/home/jovyan/jars/org.projectnessie.nessie-integrations_nessie-spark-extensions-3.4_2.12-0.99.0.jar,/home/jovyan/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar,/home/jovyan/jars/org.apache.iceberg_iceberg-spark-runtime-3.4_2.12-1.3.1.jar,/home/jovyan/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar,/home/jovyan/jars/software.amazon.awssdk_url-connection-client-2.24.8.jar,/home/jovyan/jars/org.slf4j_slf4j-api-1.7.30.jar,/home/jovyan/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar,/home/jovyan/jars/software.amazon.awssdk_utils-2.24.8.jar,/home/jovyan/jars/software.amazon.awssdk_http-client-spi-2.24.8.jar,/home/jovyan/jars/software.amazon.awssdk_annotations-2.24.8.jar


In [3]:
# Start Spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("Spark Session Started")

Spark Session Started


In [4]:
# check if all jars are available
print("----- Downloaded -----")
!ls /home/jovyan/.ivy2/jars
print("----- Local -----")
!ls /home/jovyan/jars

----- Downloaded -----
ls: cannot access '/home/jovyan/.ivy2/jars': No such file or directory
----- Local -----
com.amazonaws_aws-java-sdk-bundle-1.12.262.jar
org.apache.hadoop_hadoop-aws-3.3.4.jar
org.apache.iceberg_iceberg-spark-runtime-3.4_2.12-1.3.1.jar
org.projectnessie.nessie-integrations_nessie-spark-extensions-3.4_2.12-0.99.0.jar
org.reactivestreams_reactive-streams-1.0.4.jar
org.slf4j_slf4j-api-1.7.30.jar
org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar
software.amazon.awssdk_annotations-2.24.8.jar
software.amazon.awssdk_bundle-2.24.8.jar
software.amazon.awssdk_http-client-spi-2.24.8.jar
software.amazon.awssdk_metrics-spi-2.24.8.jar
software.amazon.awssdk_url-connection-client-2.24.8.jar
software.amazon.awssdk_utils-2.24.8.jar


In [5]:
# check size of packages
print("----- Downloaded -----")
!du -hs /home/jovyan/.ivy2/jars/*
print("----- local -----")
!du -hs /home/jovyan/jars/*

----- Downloaded -----
du: cannot access '/home/jovyan/.ivy2/jars/*': No such file or directory
----- local -----
268M	/home/jovyan/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar
944K	/home/jovyan/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar
28M	/home/jovyan/jars/org.apache.iceberg_iceberg-spark-runtime-3.4_2.12-1.3.1.jar
2.5M	/home/jovyan/jars/org.projectnessie.nessie-integrations_nessie-spark-extensions-3.4_2.12-0.99.0.jar
12K	/home/jovyan/jars/org.reactivestreams_reactive-streams-1.0.4.jar
44K	/home/jovyan/jars/org.slf4j_slf4j-api-1.7.30.jar
416K	/home/jovyan/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar
16K	/home/jovyan/jars/software.amazon.awssdk_annotations-2.24.8.jar
533M	/home/jovyan/jars/software.amazon.awssdk_bundle-2.24.8.jar
84K	/home/jovyan/jars/software.amazon.awssdk_http-client-spi-2.24.8.jar
28K	/home/jovyan/jars/software.amazon.awssdk_metrics-spi-2.24.8.jar
36K	/home/jovyan/jars/software.amazon.awssdk_url-connection-client-2.24.8.jar
212K	/home/jovyan/ja

### Create schema and tables

In [6]:
#-- 1. Création du schéma dans Nessie
spark.sql(f"""
CREATE NAMESPACE IF NOT EXISTS nessie.agence
""")

DataFrame[]

In [7]:
# Exemple : Création table client
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.agence.client (
        id INT, 
        code STRING,
        name STRING,
        actif BOOLEAN
    )
    USING iceberg
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.metadata.compression-codec'='gzip',
        'write.parquet.compression-codec'='snappy',
        'write.parquet.row-group-size-bytes' = '33554432',
        'write.target-file-size-bytes' = '33554432',
        'write.metadata.metrics.default' = 'full'
    )
""")

DataFrame[]

In [8]:
# Exemple : Création table product
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.agence.product (
        id INT, 
        code STRING,
        name STRING,
        actif BOOLEAN,
        pu FLOAT
    )
    USING iceberg
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.metadata.compression-codec'='gzip',
        'write.parquet.compression-codec'='snappy',
        'write.parquet.row-group-size-bytes' = '33554432',
        'write.target-file-size-bytes' = '33554432',
        'write.metadata.metrics.default' = 'full'
    )
""")

DataFrame[]

In [9]:
# Exemple : Création table sales
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.agence.sales (
        id INT, 
        id_client INT,
        id_product INT,
        qte INT,
        total FLOAT
    )
    USING iceberg
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.metadata.compression-codec'='gzip',
        'write.parquet.compression-codec'='snappy',
        'write.parquet.row-group-size-bytes' = '33554432',
        'write.target-file-size-bytes' = '33554432',
        'write.metadata.metrics.default' = 'full'
    )
""")

DataFrame[]

In [10]:
# check tables
spark.sql("SHOW TABLES IN nessie.agence").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   agence|   client|      false|
|   agence|  product|      false|
|   agence|    sales|      false|
+---------+---------+-----------+



### All clients.

Read all client in `raw-bucket/client` and store them inside schema `nessie.agence.client`

In [11]:
# 1. Charger toutes les données (assurez-vous d'avoir créé le bucket dans MinIO)
df_star_client = spark.read.parquet(file_star_client)
# 2. Créer une vue temporaire pour utiliser le SQL standard
df_star_client.createOrReplaceTempView("all_client_table")

In [12]:
df_star_client.select("id", "code", "name", "actif").show(3)

+---+----+---------------+-----+
| id|code|           name|actif|
+---+----+---------------+-----+
| 11|C011| Arnaud Bernard| true|
| 12|C012|Boulangerie Bio| true|
| 13|C013|    Cédric Vial| true|
+---+----+---------------+-----+
only showing top 3 rows



In [13]:
## WRITTING FROM PARQUET TO ICEBERG(NESSIE)
# On aligne le schéma sur (id, code, name, actif, pu)
df_star_client_w = df_star_client.select("id", "code", "name", "actif")

# Insertion
df_star_client_w.writeTo("nessie.agence.client").append()

In [14]:
# lecture des données et verification
# Truncate=true -> limite (char:20, colonnes tailles fixes)
spark.sql("SELECT * FROM nessie.agence.client").show(truncate=False)

+---+----+----------------+-----+
|id |code|name            |actif|
+---+----+----------------+-----+
|11 |C011|Arnaud Bernard  |true |
|12 |C012|Boulangerie Bio |true |
|13 |C013|Cédric Vial     |true |
|14 |C014|Dany Transports |true |
|15 |C015|Elena Gomez     |true |
|16 |C016|France Telecom  |true |
|17 |C017|Gérard & Co     |true |
|18 |C018|Hugo Bossis     |true |
|19 |C019|Iris Services   |true |
|20 |C020|Jean Valjean    |true |
|1  |C001|Alice Dupont    |true |
|2  |C002|Bertrand SARL   |true |
|3  |C003|Clinique du Parc|true |
|4  |C004|David Morel     |true |
|5  |C005|Ets Durand      |false|
|6  |C006|Fanny Leroy     |true |
|7  |C007|Garage Central  |true |
|8  |C008|Hélène Petit    |true |
|9  |C009|Immo Plus       |true |
|10 |C010|Julien Simon    |true |
+---+----+----------------+-----+



### All Products.

Read all product in `raw-bucket/product` and store them inside schema `nessie.agence.product`

In [16]:
from pyspark.sql.functions import col

# Définition de votre schéma cible
schema_map = {
    "id": "int",
    "code": "string",
    "name": "string",
    "actif": "boolean",
    "pu": "float"
}

In [17]:
# 1. Charger toutes les données (assurez-vous d'avoir créé le bucket dans MinIO)
df_star_product = spark.read.parquet(file_star_product)
## WRITTING FROM PARQUET TO ICEBERG(NESSIE)
# On aligne le schéma sur (id, code, name, actif, pu)
df_star_product_w = df_star_product.select([col(name).cast(dtype).alias(name) for name, dtype in schema_map.items()])
#df_star_product_w = df_star_product.select("id", "code", "name", "actif","pu")

# Insertion
df_star_product_w.writeTo("nessie.agence.product").append()

In [18]:
# lecture des données et verification
# Truncate=true -> limite (char:20, colonnes tailles fixes)
spark.sql("SELECT * FROM nessie.agence.product").show(truncate=False)

+---+----+-------------------+-----+-----+
|id |code|name               |actif|pu   |
+---+----+-------------------+-----+-----+
|11 |P011|Tablette 10"       |true |299.0|
|12 |P012|Stylet Tactile     |true |35.0 |
|13 |P013|Chargeur Rapide    |true |25.0 |
|14 |P014|Câble HDMI 2m      |true |12.0 |
|15 |P015|Batterie Externe   |true |45.0 |
|16 |P016|Sacoche Laptop     |true |30.0 |
|17 |P017|Adaptateur Wifi    |true |20.0 |
|18 |P018|Microphone USB     |true |85.0 |
|19 |P019|Projecteur LED     |true |150.0|
|20 |P020|Liseuse eBook      |true |110.0|
|1  |P001|Ordinateur Portable|true |850.0|
|2  |P002|Souris Sans Fil    |true |25.5 |
|3  |P003|Clavier Mécanique  |true |75.0 |
|4  |P004|Ecran 24 pouces    |true |150.0|
|5  |P005|Imprimante Laser   |true |210.0|
|6  |P006|Casque Audio       |true |45.0 |
|7  |P007|Disque Dur Externe |true |90.0 |
|8  |P008|Clé USB 64Go       |true |15.0 |
|9  |P009|Webcam HD          |true |55.0 |
|10 |P010|Tapis de souris    |true |10.0 |
+---+----+-

### All sales.

Read all product in `raw-bucket/sales` and store them inside schema `nessie.agence.sales`

In [19]:
from pyspark.sql.functions import col

# Définition de votre schéma cible
schema_map_sales = {
    "id": "int",
    "id_client": "int",
    "id_product": "int",
    "qte": "int",
    "total": "float"
}

In [21]:
# 1. Charger toutes les données (assurez-vous d'avoir créé le bucket dans MinIO)
df_star_sales = spark.read.parquet(file_star_sales)
df_star_sales_w = df_star_sales.select([col(name).cast(dtype).alias(name) for name, dtype in schema_map_sales.items()])
# Insertion
df_star_sales_w.writeTo("nessie.agence.sales").append()

In [22]:
spark.sql("SELECT * FROM nessie.agence.sales").show(truncate=False)

+---+---------+----------+---+-----+
|id |id_client|id_product|qte|total|
+---+---------+----------+---+-----+
|1  |1        |1         |1  |850.0|
|2  |1        |2         |2  |51.0 |
|3  |2        |3         |1  |75.0 |
|4  |2        |4         |2  |300.0|
|5  |3        |5         |1  |210.0|
|6  |3        |1         |1  |850.0|
|7  |4        |6         |3  |135.0|
|8  |4        |7         |1  |90.0 |
|9  |5        |8         |10 |150.0|
|10 |5        |9         |2  |110.0|
|11 |6        |10        |5  |50.0 |
|12 |6        |1         |1  |850.0|
|13 |7        |2         |4  |102.0|
|14 |7        |3         |2  |150.0|
|15 |8        |4         |1  |150.0|
|16 |8        |5         |1  |210.0|
|17 |9        |6         |2  |90.0 |
|18 |9        |7         |3  |270.0|
|19 |10       |8         |5  |75.0 |
|20 |10       |9         |1  |55.0 |
+---+---------+----------+---+-----+
only showing top 20 rows



In [23]:
spark.stop()